In [1]:
import os
import pandas as pd
import numpy as np

txt_label_path = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count"
txt_to_csv_path = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count\txt_to_csv"

FRAME_DUR = 0.01  # 10 ms

os.makedirs(txt_to_csv_path, exist_ok=True)

for txt_file in os.listdir(txt_label_path):
    if not txt_file.endswith(".txt"):
        continue

    txt_file_path = os.path.join(txt_label_path, txt_file)

    segments = []

    # -------------------------------
    # Read Audacity TXT
    # -------------------------------
    with open(txt_file_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 3:
                continue

            start = float(parts[0])
            end = float(parts[1])
            label = parts[2].lower()

            segments.append((start, end, label))

    if not segments:
        continue

    # -------------------------------
    # Determine total duration
    # -------------------------------
    total_duration = max(seg[1] for seg in segments)

    frame_times = np.arange(0, total_duration, FRAME_DUR)
    wheeze_labels = np.zeros(len(frame_times), dtype=int)

    # -------------------------------
    # Assign wheeze labels per frame
    # -------------------------------
    for i, t in enumerate(frame_times):
        frame_start = t
        frame_end = t + FRAME_DUR

        for seg_start, seg_end, seg_label in segments:
            if seg_label == "wheeze":
                # overlap check
                if frame_start < seg_end and frame_end > seg_start:
                    wheeze_labels[i] = 1
                    break

    # -------------------------------
    # Save CSV
    # -------------------------------
    df = pd.DataFrame({
        "frameTime": frame_times,
        "wheeze": wheeze_labels
    })

    csv_name = txt_file.replace(".txt", ".csv")
    csv_path = os.path.join(txt_to_csv_path, csv_name)

    df.to_csv(csv_path, index=False)


In [ ]:
import os
import librosa
import pandas as pd
import numpy as np

txt_label_path = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count"
wav_path = r"E:\Research Project (Prof. Preeti Rao)\Top 100 files by Wheeze count"
txt_to_csv_path = r"E:\Research Project (Prof. Preeti Rao)\Classification_24B3907\Spec_plot_amp_freq"

FRAME_DUR = 0.01  # 10 ms
WIN_DUR = 0.1     # 100 ms
SR = 16000

os.makedirs(txt_to_csv_path, exist_ok=True)

for txt_file in os.listdir(txt_label_path):
    if not txt_file.endswith(".txt"):
        continue

    base = txt_file.replace("_label_audacity.txt", "").replace(".txt", "")
    wav_file = base + ".wav"

    wav_file_path = os.path.join(wav_path, wav_file)
    txt_file_path = os.path.join(txt_label_path, txt_file)

    if not os.path.exists(wav_file_path):
        print(f"Missing WAV: {wav_file}")
        continue

    # -------------------------------
    # Load audio
    # -------------------------------
    audio, sr = librosa.load(wav_file_path, sr=SR)

    win_len = int(WIN_DUR * sr)
    hop_len = int(FRAME_DUR * sr)

    # -------------------------------
    # Read Audacity TXT
    # -------------------------------
    segments = []
    with open(txt_file_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 3:
                continue
            segments.append((float(parts[0]), float(parts[1]), parts[2].lower()))

    if not segments:
        continue

    total_duration = max(seg[1] for seg in segments)
    frame_times = np.arange(0, total_duration, FRAME_DUR)

    rows = []

    # -------------------------------
    # Frame-wise processing
    # -------------------------------
    for i, t in enumerate(frame_times):
        start_sample = int(t * sr)
        end_sample = start_sample + win_len

        if end_sample > len(audio):
            break

        window = audio[start_sample:end_sample]

        # ---------- FFT ----------
        fft = np.fft.rfft(window, n=1024)
        freqs = np.fft.rfftfreq(1024, 1/sr)
        S_mag = np.abs(fft)
        S_mag[0] = 0

        valid = (freqs >= 0) & (freqs <= 2000)
        S_mag = S_mag[valid]
        freqs = freqs[valid]

        # ---------- Features ----------
        max_idx = np.argmax(S_mag)
        max_amp = S_mag[max_idx]
        max_freq = freqs[max_idx]

        S_norm = S_mag / (np.sum(S_mag) + 1e-8)

        centroid = np.sum(freqs * S_norm)
        bandwidth = np.sqrt(np.sum((freqs - centroid) ** 2 * S_norm))
        flatness = np.exp(np.mean(np.log(S_mag + 1e-8))) / (np.mean(S_mag) + 1e-8)

        cum_energy = np.cumsum(S_norm)
        rolloff = freqs[np.where(cum_energy >= 0.85)[0][0]]

        peak_sharpness = max_amp / (np.mean(S_mag) + 1e-8)

        K = 5
        top_idx = np.argsort(S_mag)[-K:]
        peak_freq_spread = np.std(freqs[top_idx])
        peak_amp_ratio = np.max(S_mag[top_idx]) / (np.mean(S_mag[top_idx]) + 1e-8)

        # ---------- Label ----------
        wheeze = 0
        frame_end = t + FRAME_DUR
        for seg_start, seg_end, seg_label in segments:
            if seg_label == "wheeze" and t < seg_end and frame_end > seg_start:
                wheeze = 1
                break

        # ---------- Row ----------
        rows.append([
            t, wheeze,
            max_amp, max_freq,
        ])

    # -------------------------------
    # Save CSV
    # -------------------------------
    df = pd.DataFrame(rows, columns=[
        "frameTime", "wheeze",
        "maxAmp", "maxFreq",
    ])

    out_csv = os.path.join(txt_to_csv_path, txt_file.replace(".txt", ".csv"))
    df.to_csv(out_csv, index=False)